# ZAIC 2026 LifeMate — Model soup (chạy trên **L4**, không train)

Trung bình trọng số (chính xác trên delta LoRA) của các adapter cùng gốc Qwen3-4B. Tốc độ suy luận **không đổi**, Docker chỉ thay trọng số LLM.
Model soup thường giúp nhất trên dữ liệu khác phân bố train — đúng điểm yếu (65% public là app lạ).

Thử 2 công thức, cùng ASR cuối `asr_v8_large_all` (greedy):
- **A** = ½ `ft_v11_all` + ½ `ft_v11`  (cùng dữ liệu, khác tập train — ít rủi ro)
- **B** = ⅓ `ft_v11_all` + ⅓ `ft_v11` + ⅓ `ft_v12` (thêm bản train trên dữ liệu khác — đa dạng hơn)

Mốc: **52.10** (`ft_v11_all` một mình). Đổi Docker chỉ khi public **≥ 53.1**. L4 khoảng 2 giờ (~4 unit).


In [ ]:
# ==== CẤU HÌNH ====
BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
BATCH      = 32
DATA_URL   = "https://dl-challenge.zalo.ai/2026/lifemate_argfzdgert/train+public_test.zip"
DRIVE_DIR  = "/content/drive/MyDrive/ZAIC"
ASR_ALL    = DRIVE_DIR + "/asr_v8_large_all"
SOUPS = {
    "A": [DRIVE_DIR + "/ft_v11_all_adapter", DRIVE_DIR + "/ft_v11_adapter"],
    "B": [DRIVE_DIR + "/ft_v11_all_adapter", DRIVE_DIR + "/ft_v11_adapter", DRIVE_DIR + "/ft_v12_adapter"],
}
RUN = ["A", "B"]          # bớt "B" nếu muốn tiết kiệm (~40 phút)
WORK, DATA_RAW, RES = "/content/work", "/content/data_raw", "/content/results_soup"
print("RUN:", RUN)


In [ ]:
# ==== KIỂM TRA ĐÚNG FILE (chạy đầu tiên) ====
NOTEBOOK = "soup"
from google.colab import drive; drive.mount("/content/drive")
import os, zipfile
_z = os.path.join(DRIVE_DIR, "lifemate_baseline.zip")
assert os.path.exists(_z), "Drive không có MyDrive/ZAIC/lifemate_baseline.zip"
_v = zipfile.ZipFile(_z).read("VERSION").decode().strip() if "VERSION" in zipfile.ZipFile(_z).namelist() else "(không có VERSION — zip cũ)"
assert _v == "2026-11-14-1400", f"lifemate_baseline.zip trên Drive là bản {_v}, cần bản 2026-11-14-1400 → ghi đè bằng bản mới nhất (zip của repo)"
_need = ["asr_v8_large_all/config.json"] + sorted({a.split("/ZAIC/")[1] + "/adapter_config.json" for r in RUN for a in SOUPS[r]})
_miss = [p for p in _need if not os.path.exists(os.path.join(DRIVE_DIR, p))]
assert not _miss, "Thiếu trên Drive: " + ", ".join(_miss)
print(f"OK: notebook {NOTEBOOK} | zip {_v} | đủ: {_need}")


In [ ]:
import subprocess, torch, os
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "Chưa bật GPU"
DTYPE = "bfloat16" if torch.cuda.get_device_capability()[0] >= 8 else "float16"
GPU_NAME = torch.cuda.get_device_name(0); print(GPU_NAME, DTYPE)
os.makedirs(RES, exist_ok=True)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, zipfile, shutil, glob, subprocess, json, time
shutil.rmtree(WORK, ignore_errors=True); zipfile.ZipFile(os.path.join(DRIVE_DIR, "lifemate_baseline.zip")).extractall(WORK)
assert os.path.exists(f"{WORK}/scripts/detcheck.py"), "lifemate_baseline.zip trong Drive là bản cũ -> upload bản mới"
data_zip = os.path.join(DRIVE_DIR, "train+public_test.zip")
if not os.path.exists(data_zip):
    data_zip = "/content/train+public_test.zip"
    if not os.path.exists(data_zip):
        assert subprocess.run(["wget", "-q", "-O", data_zip, DATA_URL]).returncode == 0, "Tải dữ liệu lỗi"
if not glob.glob(DATA_RAW + "/**/train/train.jsonl", recursive=True):
    zipfile.ZipFile(data_zip).extractall(DATA_RAW)
DATA = os.path.dirname(os.path.dirname(glob.glob(DATA_RAW + "/**/train/train.jsonl", recursive=True)[0]))
print("DATA:", DATA)
assert os.path.exists(f"{WORK}/lifemate/soup.py"), "lifemate_baseline.zip chưa có lifemate/soup.py -> ghi đè bản mới nhất"

In [ ]:
%%capture pipout
!pip install -q vllm peft soundfile
!pip uninstall -y torchao

In [ ]:
print(pipout.stderr[-800:])
!pip list 2>/dev/null | grep -iE "^(vllm|torch|transformers|peft) " | tee {RES}/versions.txt
%cd {WORK}
!cat splits/val_unseen.txt splits/val_seen.txt > splits/val_all.txt

## 1. ASR public một lần (greedy, giống bản 52.10)


In [ ]:
%cd {WORK}
t0 = time.time()
!python -m lifemate.asr --root {DATA} --split public_test --model {ASR_ALL} --dtype {DTYPE} --out asr_public.json 2>&1 | tail -1
print(f"ASR public: {time.time() - t0:.0f}s")


## 2. Mỗi công thức: trộn → chạy public → validate


In [ ]:
%cd {WORK}
subs = {}
for R in RUN:
    LLM = f"/content/soup_{R}/merged"
    ads = " ".join(SOUPS[R])
    !python -m lifemate.soup --base {BASE_MODEL} --adapters {ads} --out {LLM} 2>&1 | grep -E "^soup|Error|error" | tail -3
    assert os.path.exists(f"{LLM}/config.json"), f"soup {R} lỗi — xem log"
    !python -m lifemate.run --root {DATA} --split public_test --backend vllm --model {LLM} --dtype {DTYPE} --batch {BATCH} --transcripts asr_public.json --out runs/soup_{R} 2>&1 | tail -1
    SUB = f"submission_public_soup{R}.jsonl"
    shutil.copy(f"runs/soup_{R}/submission.jsonl", f"{RES}/{SUB}")
    shutil.copy(f"runs/soup_{R}/submission.jsonl", os.path.join(DRIVE_DIR, SUB))
    shutil.copy(f"{LLM}/soup.json", f"{RES}/soup_{R}.json")
    subs[R] = SUB
    shutil.rmtree(LLM, ignore_errors=True)      # giải phóng ~8GB đĩa
    print(f"== soup {R} xong -> NỘP FILE: {os.path.join(DRIVE_DIR, SUB)}")
%cd {DATA}/public_test
for R, SUB in subs.items():
    !python -m zaic_tools --split public_test --root .. --validate {RES}/{SUB} 2>&1 | tail -1 | tee {RES}/validate_{R}.txt


## 3. Đóng gói


In [ ]:
z = shutil.make_archive("/content/results_soup", "zip", RES)
shutil.copy(z, os.path.join(DRIVE_DIR, "results_soup.zip")); print("saved results_soup.zip trên Drive | file nộp:", subs)
